In [1]:
from google.colab import files
uploaded = files.upload()

Saving Sample-Raw-No-Punctuations.zip to Sample-Raw-No-Punctuations.zip


In [2]:
import zipfile
import os

zip_path = "/content/Sample-Raw-No-Punctuations.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

files = os.listdir(extract_path)
print(files)

['Sample-Raw-No-Punctuations']


In [3]:
import os

DATA_DIR = "/content/dataset/Sample-Raw-No-Punctuations"

for file in sorted(os.listdir(DATA_DIR)):
    print(file)

sample-1.txt
sample-2.txt
sample-3.txt
sample-4.txt
sample-5.txt
sample-6.txt
sample-7.txt


In [4]:
!pip install -q transformers sentence-transformers sentencepiece accelerate sacremoses

In [5]:
!nvidia-smi

Wed Nov 11 15:24:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Total GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

CUDA available: True
GPU: Tesla T4
Total GPU memory: 14.56 GB


In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

PUNCT_MODEL = "Horizon-Labs/punctuation-restoration-small"

punct_tokenizer = AutoTokenizer.from_pretrained(PUNCT_MODEL)
punct_model = AutoModelForTokenClassification.from_pretrained(PUNCT_MODEL)

punct_model = punct_model.to("cuda")
punct_model.eval()

print("Punctuation model loaded successfully!")

config.json:   0%|          | 0.00/2.11k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  563MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Punctuation model loaded successfully!


In [8]:
print("Model labels:", punct_model.config.id2label)

print("GPU memory used:",
      round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")

Model labels: {0: '0', 1: '.', 2: ',', 3: '?', 4: '-', 5: ':'}
GPU memory used: 0.525 GB


In [9]:
import torch

def restore_punctuation(text):
    words = text.split()
    result = []

    chunk_size = 200

    for start in range(0, len(words), chunk_size):
        chunk = words[start:start + chunk_size]

        inputs = punct_tokenizer(
            chunk,
            is_split_into_words=True,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        word_ids = inputs.word_ids()

        model_inputs = {
            k: v.to("cuda") for k, v in inputs.items()
            if k in ["input_ids", "attention_mask", "token_type_ids"]
        }

        with torch.no_grad():
            logits = punct_model(**model_inputs).logits

        predictions = logits.argmax(dim=-1)[0].cpu().tolist()

        last_subtoken = {}

        for token_idx, word_id in enumerate(word_ids):
            if word_id is not None:
                last_subtoken[word_id] = predictions[token_idx]

        for i, word in enumerate(chunk):
            label = punct_model.config.id2label[
                last_subtoken.get(i, 0)
            ]

            if label != "0":
                word += label

            result.append(word)

    return " ".join(result)


sample_path = os.path.join(DATA_DIR, "sample-1.txt")

with open(sample_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

punctuated_text = restore_punctuation(raw_text)

print("ORIGINAL TEXT:\n")
print(raw_text[:1000])

print("\nPUNCTUATED TEXT:\n")
print(punctuated_text[:1000])

ORIGINAL TEXT:

भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है स्टेशन का नाम कांजीवरम जरूर है किंतु नगर का नाम कांचीपुरम है यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का यह पवित्र नगर मथुरा है दक्षिण रेलवे के चेन्नै स्टेशन से धनुष्कोटि जाने वाली मुख्य लाइन पर चेन्नै से 35 मील की दूरी पर चेंगलपट स्टेशन है ये सातों धर्मस्थल सात नगरों या सप्तपुरियों के रूप में ग्रंथों में वर्णित हैं ऐसा कहा गया है कि चतुर्मास में इन सप्तपुरियों का दर्शन मोक्ष प्रदान करने वाला होता है हिंदू धर्म में तीर्थ का बड़ा महत्व है पूर्वोत्तर रेलवे की एक लाइन भटनी से NULL

In [10]:
import os

OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

punctuated_data = {}

for filename in sorted(os.listdir(DATA_DIR)):
    if filename.endswith(".txt"):
        path = os.path.join(DATA_DIR, filename)

        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        result = restore_punctuation(text)
        punctuated_data[filename] = result

        output_path = os.path.join(OUTPUT_DIR, filename)

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(result)

        print(filename, "completed")

print("\nAll 7 files processed!")

sample-1.txt completed
sample-2.txt completed
sample-3.txt completed
sample-4.txt completed
sample-5.txt completed
sample-6.txt completed
sample-7.txt completed

All 7 files processed!


In [11]:
from sentence_transformers import SentenceTransformer

ORDER_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

order_model = SentenceTransformer(
    ORDER_MODEL,
    device="cuda"
)

print("Sentence ordering model loaded!")
print("GPU memory:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence ordering model loaded!
GPU memory: 0.971 GB


In [12]:
import re
import os

sentences_by_file = {}

for filename in sorted(punctuated_data):
    text = punctuated_data[filename]
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    sentences_by_file[filename] = sentences
    print(filename, ":", len(sentences), "sentences")

print("\nSentence extraction completed!")

sample-1.txt : 50 sentences
sample-2.txt : 52 sentences
sample-3.txt : 51 sentences
sample-4.txt : 50 sentences
sample-5.txt : 52 sentences
sample-6.txt : 54 sentences
sample-7.txt : 63 sentences

Sentence extraction completed!


In [13]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def reorder_sentences(sentences):
    if len(sentences) <= 1:
        return sentences

    embeddings = order_model.encode(
        sentences,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    similarity = cosine_similarity(embeddings)
    n = len(sentences)

    best_order = None
    best_score = -float("inf")

    for start in range(n):
        remaining = set(range(n))
        remaining.remove(start)
        order = [start]
        score = 0
        current = start

        while remaining:
            next_idx = max(remaining, key=lambda j: similarity[current][j])
            score += similarity[current][next_idx]
            order.append(next_idx)
            remaining.remove(next_idx)
            current = next_idx

        if score > best_score:
            best_score = score
            best_order = order

    return [sentences[i] for i in best_order]


ordered_data = {}

for filename, sentences in sentences_by_file.items():
    ordered = reorder_sentences(sentences)
    ordered_data[filename] = ordered

    output_path = os.path.join(OUTPUT_DIR, filename.replace(".txt", "_ordered.txt"))
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(ordered))

    print(filename, "reordered")

print("\nAll files reordered!")

sample-1.txt reordered
sample-2.txt reordered
sample-3.txt reordered
sample-4.txt reordered
sample-5.txt reordered
sample-6.txt reordered
sample-7.txt reordered

All files reordered!


In [14]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

del order_model
gc.collect()
torch.cuda.empty_cache()

TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-hi-en"

translation_tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATION_MODEL)
translation_model = translation_model.to("cuda")
translation_model.eval()

print("Translation model loaded!")
print("GPU memory:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  304MB            

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  304MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Translation model loaded!
GPU memory: 0.817 GB


In [15]:
import torch
import os

def translate_text(text):
    inputs = translation_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        output = translation_model.generate(
            **inputs,
            max_length=512,
            num_beams=2
        )

    return translation_tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )


translated_data = {}

for filename, sentences in ordered_data.items():
    groups = [
        sentences[i:i + 5]
        for i in range(0, len(sentences), 5)
    ]

    translations = []

    for group in groups:
        hindi_text = " ".join(group)
        english_text = translate_text(hindi_text)
        translations.append(english_text)

    translated_data[filename] = translations

    output_path = os.path.join(
        OUTPUT_DIR,
        filename.replace(".txt", "_translated.txt")
    )

    with open(output_path, "w", encoding="utf-8") as f:
        for i, translation in enumerate(translations, 1):
            f.write(f"Group {i}:\n{translation}\n\n")

    print(filename, "translated:", len(translations), "groups")

print("\nAll files translated!")

sample-1.txt translated: 10 groups
sample-2.txt translated: 11 groups
sample-3.txt translated: 11 groups
sample-4.txt translated: 10 groups
sample-5.txt translated: 11 groups
sample-6.txt translated: 11 groups
sample-7.txt translated: 13 groups

All files translated!


In [16]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/NLP_Project_Outputs",
    "zip",
    OUTPUT_DIR
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>